# COSMOLOGY · Interactive Physics Atlas
**Baumann, *Cosmology* · ΛCDM → Inflation → Perturbations → Primordial GWs → CMB**


In [2]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import cumulative_trapezoid, solve_ivp
from scipy.optimize import brentq
from IPython.display import display
import io

try:
    import ipywidgets as widgets
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False

plt.rcParams.update({
    "figure.figsize": (10, 4.8), "figure.dpi": 105, "savefig.dpi": 160,
    "axes.titlesize": 14, "axes.labelsize": 11, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": .20, "lines.linewidth": 2.35,
})
COL = {"radiation":"#D19A36","matter":"#347EAF","vacuum":"#8D65B5",
       "tensor":"#B35473","teal":"#278B85","dark":"#263A4A","other":"#8C99A4"}

# Show functional ipywidgets on a configured local Jupyter; otherwise save a static preview.
def lab(fn, controls, defaults):
    if HAS_WIDGETS:
        w = widgets.interactive(fn, **controls)
        w.layout.width = '100%'
        display(w)
    else:
        print('Static preview (interactive mode requires ipywidgets).')
        fn(**defaults)
print('Python environment: NumPy, Matplotlib, SciPy | widgets:', 'ready' if HAS_WIDGETS else 'not installed')

Python environment: NumPy, Matplotlib, SciPy | widgets: ready


## 01 · ΛCDM: ποιο συστατικό κυριαρχεί;
$$
\boxed{E^2(a)=\Omega_{r0}a^{-4}+\Omega_{m0}a^{-3}+\Omega_{\Lambda0}},\qquad
\boxed{\Omega_i(a)=\frac{\Omega_{i0}a^{-3(1+w_i)}}{E^2(a)}}
$$
$$\boxed{q(a)=\frac{\Omega_r(a)}{1}+\frac{\Omega_m(a)}2-\Omega_\Lambda(a)}$$
**Βλέπουμε:** radiation → matter → Λ και πότε η διαστολή γίνεται επιταχυνόμενη ($q<0$). Επίπεδο μοντέλο, $\Omega_{r0}=9\cdot10^{-5}$.

In [5]:
def lcdm_lab(Om0=0.32):
    Or0=9e-5; Ol0=1-Om0-Or0
    a=np.geomspace(1e-6,3.0,1200)
    e2=Or0/a**4+Om0/a**3+Ol0
    fr, fm, fl=(Or0/a**4)/e2,(Om0/a**3)/e2,Ol0/e2
    q=fr+0.5*fm-fl
    aeq=Or0/Om0
    aacc=brentq(lambda x: (2*Or0/x**4+Om0/x**3-2*Ol0),0.1,2)
    fig,axes=plt.subplots(2,1,figsize=(10,7.5),sharex=True,gridspec_kw={'height_ratios':[1.4,1]})
    axes[0].semilogx(a,fr,color=COL['radiation'],label=r'$\Omega_r(a)$')
    axes[0].semilogx(a,fm,color=COL['matter'],label=r'$\Omega_m(a)$')
    axes[0].semilogx(a,fl,color=COL['vacuum'],label=r'$\Omega_\Lambda(a)$')
    axes[0].axvline(aeq,ls=':',color=COL['other'],label='matter–radiation equality')
    axes[0].axvline(1,ls='--',color=COL['dark'],alpha=.6)
    axes[0].set(ylabel=r'Density fraction $\Omega_i(a)$',ylim=(-.02,1.05))
    axes[0].legend(loc='center left',fontsize=9)
    axes[1].semilogx(a,q,color=COL['teal'],label=r'$q(a)$')
    axes[1].axhline(0,color='k',lw=1)
    axes[1].axvline(aacc,color=COL['tensor'],ls='--',label=rf'$q=0,\ z={1/aacc-1:.2f}$')
    axes[1].axvline(1,ls=':',color=COL['dark'])
    axes[1].set(xlabel=r'Scale factor $a$ (today: 1)',ylabel=r'Deceleration $q$',ylim=(-1.1,1.1))
    axes[1].legend(loc='upper right',fontsize=9)
    fig.tight_layout();plt.show()
    print(f'Ωm0={Om0:.3f}, ΩΛ0={Ol0:.5f} | a_eq={aeq:.5g} | acceleration begins at a={aacc:.3f}, z={1/aacc-1:.3f}')
lab(lcdm_lab, {'Om0':widgets.FloatSlider(value=.32,min=.2,max=.45,step=.01,description='Ωm today',continuous_update=False)} if HAS_WIDGETS else {}, {'Om0':.32})

interactive(children=(FloatSlider(value=0.32, continuous_update=False, description='Ωm today', max=0.45, min=0…

## 02 · Inflation: γιατί μικραίνει η comoving ακτίνα Hubble;
$$
\boxed{r_H=(aH)^{-1}},\qquad
\boxed{\frac{r_H(N)}{r_{H,i}}=e^{\frac{1+3w}{2}N}},\qquad
\boxed{\frac{1}{k}=\mathrm{const.}}
$$
**Βλέπουμε:** Hubble crossing όταν $r_H=1/k$. Το $1/k$ είναι η ανηγμένη συνεκτεινόμενη κλίμακα του mode (όχι $2\pi/k$). Κανονικοποιούμε $a_iH_i=1$.

In [8]:
def hubble_lab(w=-0.95,k=8.0,Nmax=7.0):
    n=np.linspace(0,Nmax,700)
    rh=np.exp(.5*(1+3*w)*n)
    cross=-2*np.log(k)/(1+3*w) if abs(1+3*w)>1e-8 else np.nan
    fig,ax=plt.subplots()
    ax.semilogy(n,rh,color=COL['tensor'],label=r'$r_H/r_{H,i}$')
    ax.axhline(1/k,color=COL['matter'],ls='--',label=rf'mode scale $1/k={1/k:.3f}$')
    if 0<=cross<=Nmax:
        ax.axvline(cross,color=COL['teal'],ls=':')
        ax.scatter([cross],[1/k],s=68,c=COL['teal'],zorder=10,label=rf'crossing $N={cross:.2f}$')
    ax.set(xlabel='E-folds N',ylabel='Comoving length (relative to initial Hubble radius)',title='Hubble exit / re-entry: crossing of scales')
    ax.legend();fig.tight_layout();plt.show()
    print('Inflation (w < −1/3):',w < -1/3,'| ε_H =',round(1.5*(1+w),3),'| crossing:', round(cross,3) if np.isfinite(cross) else 'not defined')
lab(hubble_lab,{'w':widgets.FloatSlider(value=-.95,min=-1,max=.5,step=.05,description='w',continuous_update=False),
               'k':widgets.FloatSlider(value=8,min=.25,max=16,step=.25,description='k / (aiHi)',continuous_update=False),
               'Nmax':widgets.FloatSlider(value=7,min=2,max=10,step=1,description='N max',continuous_update=False)} if HAS_WIDGETS else {}, {'w':-.95,'k':8.,'Nmax':7.})

interactive(children=(FloatSlider(value=-0.95, continuous_update=False, description='w', max=0.5, min=-1.0, st…

## 03 · Inflation: $V(\phi)$ και slow-roll
$$
\boxed{\rho_\phi=\tfrac12\dot\phi^2+V,\quad p_\phi=\tfrac12\dot\phi^2-V}
$$
$$
\boxed{3H\dot\phi\simeq-V_{,\phi}},\quad
\boxed{\epsilon_V=\frac{M_{\rm Pl}^2}{2}\left(\frac{V_{,\phi}}V\right)^2},\quad
\boxed{\eta_V=M_{\rm Pl}^2\frac{V_{,\phi\phi}}V}
$$
**Βλέπουμε:** κλίση δυναμικού, $\epsilon_V$, $\eta_V$ και τη θέση του inflaton. Ο άξονας $\phi$ **δεν είναι χρόνος**. Δυναμικά: plateau ή quadratic.

In [11]:
b=np.sqrt(2/3)
def potential(x, model):
    x=np.asarray(x)
    return (1-np.exp(-b*x))**2 if model=='Plateau' else .5*x**2

def potential_eps_eta(x,model):
    x=np.asarray(x)
    if model=='Plateau':
        y=np.exp(-b*x)
        return (4/3)*(y/(1-y))**2, (4/3)*y*(2*y-1)/(1-y)**2
    return 2/x**2, 2/x**2

def phi_end(model):
    return np.log(1+2/np.sqrt(3))/b if model=='Plateau' else np.sqrt(2)

def remaining_efolds(x,model):
    end=phi_end(model)
    if x<=end: return 0.0
    return ((np.exp(b*x)-np.exp(b*end))/(2*b*b) - (x-end)/(2*b)) if model=='Plateau' else (x*x-end*end)/4

def potential_lab(model='Plateau',phi=5.8):
    x=np.linspace(.25,18,1800); eps,eta=potential_eps_eta(x,model)
    ep,et=potential_eps_eta(phi,model)
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(11.5,4.8))
    y=potential(x,model)
    ax1.plot(x,y,color=COL['matter']);ax1.scatter([phi],[potential(phi,model)],color=COL['tensor'],s=65,zorder=5)
    ax1.axvline(phi,color=COL['tensor'],ls=':')
    ax1.axvline(phi_end(model),color=COL['other'],ls='--',label=r'approx. $\epsilon_V=1$')
    ax1.set(xlabel=r'$\phi/M_{\rm Pl}$',ylabel=r'$V(\phi)/V_0$',title=f'{model} potential',xlim=(0,18))
    ax1.legend(fontsize=8)
    ax2.semilogy(x,eps,color=COL['tensor'],label=r'$\epsilon_V$')
    ax2.semilogy(x,np.maximum(np.abs(eta),1e-7),color=COL['teal'],label=r'$|\eta_V|$')
    ax2.axvline(phi,color=COL['dark'],ls=':');ax2.axhline(1,color=COL['other'],ls='--')
    ax2.set(xlabel=r'$\phi/M_{\rm Pl}$',ylabel='Slow-roll parameters',title='Potential slow-roll',ylim=(1e-6,100))
    ax2.legend(fontsize=9)
    fig.tight_layout();plt.show()
    print(f'phi/Mpl={phi:.2f} | epsilon_V={ep:.4g} | eta_V={et:.4g} | approx. e-folds remaining={remaining_efolds(phi,model):.2f}')
lab(potential_lab,{'model':widgets.ToggleButtons(options=['Plateau','Quadratic'],value='Plateau',description='Potential'),
                   'phi':widgets.FloatSlider(value=5.8,min=.5,max=17,step=.1,description='φ/Mpl',continuous_update=False)} if HAS_WIDGETS else {}, {'model':'Plateau','phi':5.8})

interactive(children=(ToggleButtons(description='Potential', options=('Plateau', 'Quadratic'), value='Plateau'…

## 04 · Slow-roll trajectory: το πεδίο κυλά προς το τέλος του Inflation
$$
\boxed{\frac{d(\phi/M_{\rm Pl})}{dN}\simeq-\frac{d\ln V}{d(\phi/M_{\rm Pl})}},\qquad
\boxed{\epsilon_V(\phi_{\rm end})\simeq1}
$$
**Βλέπουμε:** $\phi(N)$ και $\epsilon_V(N)$ από αριθμητική ολοκλήρωση της **slow-roll approximation** (όχι πλήρες KG dynamics). Το μοντέλο παύει στο $\epsilon_V=1$.

In [14]:
def trajectory_lab(model='Plateau',phi_initial=5.7):
    end=phi_end(model)
    x0=max(phi_initial,end+0.03)
    def rhs(n,y):
        x=float(y[0])
        return [-2*b/(np.exp(b*x)-1) if model=='Plateau' else -2/x]
    def hit_end(n,y): return y[0]-end
    hit_end.terminal=True;hit_end.direction=-1
    solution=solve_ivp(rhs,(0,160),[x0],events=hit_end,rtol=1e-8,atol=1e-10,max_step=.15)
    ns=solution.t; xs=solution.y[0]
    eps,_=potential_eps_eta(xs,model)
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(11.5,4.2))
    ax1.plot(ns,xs,color=COL['matter']); ax1.axhline(end,color=COL['other'],ls='--')
    ax1.set(xlabel='Elapsed e-folds N',ylabel=r'$\phi/M_{\rm Pl}$',title='Slow-roll field trajectory')
    ax2.semilogy(ns,eps,color=COL['tensor']);ax2.axhline(1,color=COL['dark'],ls='--')
    ax2.set(xlabel='Elapsed e-folds N',ylabel=r'$\epsilon_V(N)$',title='End of inflation',ylim=(max(min(eps)*.5,1e-7),2))
    fig.tight_layout();plt.show()
    if solution.t_events[0].size: print(f'Approximate end after {solution.t_events[0][0]:.2f} e-folds; phi_end={end:.3f} Mpl')
    else: print('Integration window ends before epsilon_V reaches 1. Increase duration for large phi_initial.')
lab(trajectory_lab,{'model':widgets.ToggleButtons(options=['Plateau','Quadratic'],description='Potential'),
                    'phi_initial':widgets.FloatSlider(value=5.7,min=2,max=17,step=.1,description='φ initial',continuous_update=False)} if HAS_WIDGETS else {}, {'model':'Plateau','phi_initial':5.7})

interactive(children=(ToggleButtons(description='Potential', options=('Plateau', 'Quadratic'), value='Plateau'…

## 05 · Scalar Fourier modes
$$
\boxed{\delta(x)=A_1\cos(k_1x)+A_2\cos(k_2x)},\qquad
\boxed{\lambda_{\rm com}=\frac{2\pi}{k}}
$$
**Βλέπουμε:** μεγαλύτερο $k$ → μικρότερο συνεκτεινόμενο μήκος κύματος· τα modes αθροίζονται γραμμικά.

In [17]:
def fourier_lab(k1=2,k2=7,A2=0.45):
    x=np.linspace(0,2*np.pi,800);y1=np.cos(k1*x); y2=A2*np.cos(k2*x)
    fig,ax=plt.subplots(figsize=(10,4.2))
    ax.plot(x,y1,color=COL['matter'],lw=1.6,alpha=.8,label=rf'$\cos({k1}x)$')
    ax.plot(x,y2,color=COL['radiation'],lw=1.6,alpha=.8,label=rf'${A2:.2f}\cos({k2}x)$')
    ax.plot(x,y1+y2,color=COL['tensor'],lw=2.5,label='sum')
    ax.set(xlabel='Comoving x (normalized)',ylabel=r'$\delta(x)$',title='Superposition of two Fourier modes',xlim=(0,2*np.pi))
    ax.legend();fig.tight_layout();plt.show()
lab(fourier_lab,{'k1':widgets.IntSlider(value=2,min=1,max=10,description='k₁'),
                'k2':widgets.IntSlider(value=7,min=1,max=14,description='k₂'),
                'A2':widgets.FloatSlider(value=.45,min=0,max=1,step=.05,description='A₂')} if HAS_WIDGETS else {}, {'k1':2,'k2':7,'A2':.45})

interactive(children=(IntSlider(value=2, description='k₁', max=10, min=1), IntSlider(value=7, description='k₂'…

## 06 · Primordial scalar & tensor power spectra
$$
\boxed{\mathcal{P}_{\mathcal{R}}(k)=A_s(k/k_*)^{n_s-1}},\quad
\boxed{\mathcal{P}_t(k)=rA_s(k/k_*)^{n_t}},\quad
\boxed{n_t\simeq-r/8}
$$
$$\boxed{r=\mathcal{P}_t(k_*)/\mathcal{P}_{\mathcal{R}}(k_*)\simeq16\epsilon_*},\qquad
\boxed{\mathcal{P}_t\simeq\frac{2H_*^2}{\pi^2M_{\rm Pl}^2}}$$
**Βλέπουμε:** ο $n_s$ αλλάζει την κλίση των scalars· ο $r$ αλλάζει το tensor πλάτος. Μονοπεδιακό slow-roll, $A_s=2.1\times10^{-9}$, $k_*=0.05\,\mathrm{Mpc}^{-1}$, θεωρητικό φάσμα (όχι ανίχνευση tensors).

In [20]:
def primordial_lab(ns=.965,r=.03):
    As=2.1e-9; pivot=.05; k=np.geomspace(1e-4,10,800)
    pr=As*(k/pivot)**(ns-1); nt=-r/8; pt=r*As*(k/pivot)**nt
    fig,ax=plt.subplots(figsize=(10,4.8))
    ax.loglog(k,pr,color=COL['matter'],label=r'$\mathcal{P}_{\mathcal{R}}(k)$')
    ax.loglog(k,pt,color=COL['tensor'],label=r'$\mathcal{P}_t(k)$')
    ax.axvline(pivot,color=COL['other'],ls='--',label=r'$k_*=0.05\,\mathrm{Mpc}^{-1}$')
    ax.set(xlabel=r'Comoving wavenumber $k$ [Mpc$^{-1}$]',ylabel='Dimensionless power',title='Primordial scalar and tensor spectra')
    ax.legend();fig.tight_layout();plt.show()
    H_over_Mpl=np.pi*np.sqrt(As*r/2)
    print(f'ns={ns:.3f} | r={r:.3f} | nt≈{nt:.5f} | ε*≈{r/16:.5f} | H*/Mpl≈{H_over_Mpl:.3g}')
lab(primordial_lab,{'ns':widgets.FloatSlider(value=.965,min=.92,max=1.04,step=.005,description='nₛ',continuous_update=False),
                    'r':widgets.FloatSlider(value=.03,min=.001,max=.15,step=.001,description='r',continuous_update=False)} if HAS_WIDGETS else {}, {'ns':.965,'r':.03})

interactive(children=(FloatSlider(value=0.965, continuous_update=False, description='nₛ', max=1.04, min=0.92, …

## 07 · Primordial GWs: superhorizon freeze-out & subhorizon damping
$$
\boxed{h_k''+2\mathcal{H} h_k'+k^2h_k=0},\qquad
\boxed{\mu_k=ah_k:\ \mu_k''+(k^2-a''/a)\mu_k=0}
$$
$$
\boxed{h_k^{\mathrm{rad}}=h_i\frac{\sin(k\eta)}{k\eta}},\qquad
\boxed{h_k^{\mathrm{mat}}=3h_i\frac{\sin(k\eta)-k\eta\cos(k\eta)}{(k\eta)^3}}
$$
**Βλέπουμε:** $k\eta\ll1$: $h_k\approx$ σταθερό· $k\eta\gg1$: ταλάντωση με φθίνον πλάτος. Ακριβείς ιδανικές λύσεις **κατά την ακτινοκρατία / ύλη-κυριαρχία**, χωρίς ανισότροπες πηγές.

In [23]:
def tensor_transfer_lab(era='Radiation',k=1.5):
    eta=np.linspace(.005,14,1600); x=k*eta
    rad=np.sinc(x/np.pi)
    mat=np.where(np.abs(x)<1e-2,1-x*x/10 + x**4/280, 3*(np.sin(x)-x*np.cos(x))/x**3)
    y=rad if era=='Radiation' else mat
    eta_cross=(1 if era=='Radiation' else 2)/k
    fig,ax=plt.subplots(figsize=(10,4.9))
    ax.plot(eta,y,color=COL['tensor'],label=rf'$h_k/h_i$ ({era})')
    ax.axvline(eta_cross,color=COL['teal'],ls='--',label=rf'$k=\mathcal{{H}}$ at $\eta\approx{eta_cross:.2f}$')
    ax.axhline(0,color='k',lw=.7)
    ax.set(xlabel='Conformal time η (normalized)',ylabel=r'$h_k/h_i$',title='GW transfer function: freeze-out → oscillation',ylim=(-.5,1.09))
    ax.legend();fig.tight_layout();plt.show()
    print('Re-entry criterion: k=𝓗,  𝓗=1/η (radiation), 2/η (matter).')
lab(tensor_transfer_lab,{'era':widgets.ToggleButtons(options=['Radiation','Matter'],description='Era'),
                         'k':widgets.FloatSlider(value=1.5,min=.5,max=3,step=.25,description='k',continuous_update=False)} if HAS_WIDGETS else {}, {'era':'Radiation','k':1.5})

interactive(children=(ToggleButtons(description='Era', options=('Radiation', 'Matter'), value='Radiation'), Fl…

## 08 · Οι δύο tensor πολώσεις $+$ και $\times$
$$
\boxed{h_{ij}=\begin{pmatrix}h_+&h_\times&0\\h_\times&-h_+&0\\0&0&0\end{pmatrix}}
$$
**Βλέπουμε:** μια αρχικά κυκλική διάταξη δοκιμαστικών μαζών παραμορφώνεται κάθετα στη διάδοση. Η παραμόρφωση είναι **σκόπιμα υπερβολική** για να φαίνεται.

In [26]:
def polarization_lab(phase=0.,mix=0.,amplitude=.3):
    theta=np.linspace(0,2*np.pi,300)
    circle=np.vstack([np.cos(theta),np.sin(theta)])
    ang=np.linspace(0,2*np.pi,16,endpoint=False)
    ring=np.vstack([np.cos(ang),np.sin(ang)])
    hp=amplitude*np.cos(phase)*np.cos(mix*np.pi/2)
    hx=amplitude*np.cos(phase)*np.sin(mix*np.pi/2)
    h=np.array([[hp,hx],[hx,-hp]])
    p=np.eye(2)+0.5*h
    fig,ax=plt.subplots(figsize=(6,5.5))
    ax.plot(*circle,ls='--',color=COL['other'],label='Undisturbed circle')
    ax.plot(*(p@circle),color=COL['tensor'],label='GW-distorted ring')
    displaced=p@ring
    ax.scatter(*displaced,s=30,color=COL['tensor']);ax.scatter(*ring,s=12,color=COL['other'])
    for i in range(ring.shape[1]):
        ax.plot([ring[0,i],displaced[0,i]],[ring[1,i],displaced[1,i]],color=COL['teal'],lw=.8,alpha=.7)
    ax.set(xlim=(-1.22,1.22),ylim=(-1.22,1.22),aspect='equal',title='Transverse tidal deformation',xlabel='x',ylabel='y')
    ax.legend(loc='upper right',fontsize=8);fig.tight_layout();plt.show()
    print(f'h_plus={hp:.3f}, h_cross={hx:.3f} | schematic, not observational amplitudes')
lab(polarization_lab,{'phase':widgets.FloatSlider(value=0,min=0,max=2*np.pi,step=np.pi/12,description='Phase',continuous_update=False),
                      'mix':widgets.FloatSlider(value=0,min=0,max=1,step=.1,description='0:+  1:×',continuous_update=False),
                      'amplitude':widgets.FloatSlider(value=.3,min=.05,max=.5,step=.05,description='strain (toy)',continuous_update=False)} if HAS_WIDGETS else {}, {'phase':0.,'mix':0.,'amplitude':.3})

interactive(children=(FloatSlider(value=0.0, continuous_update=False, description='Phase', max=6.2831853071795…

## 09 · Παρατηρησιακό $C_\ell$: **Planck 2018 TT**
$$
\boxed{\Theta(\hat n)=\frac{T(\hat n)-\bar T}{\bar T}},\quad
\boxed{D_\ell^{TT}=\frac{\ell(\ell+1)}{2\pi}C_\ell^{TT}},\quad
\boxed{C_\ell^{TT}=\bar T^2 C_\ell^{\Theta\Theta}}
$$
**Βλέπουμε:** πραγματικές μετρήσεις $D_\ell^{TT}$ σε $\mu\mathrm K^2$ με $1\sigma$ σφάλματα και το **δημοσιευμένο, binned** best-fit ΛCDM. Acoustic peaks · $\theta\sim180^\circ/\ell$.

**Πηγή:** [Planck PR3, NASA/IPAC IRSA — `COM_PowerSpect_CMB-TT-binned_R3.01.txt`](https://irsa.ipac.caltech.edu/data/Planck/release_3/ancillary-data/cosmoparams/COM_PowerSpect_CMB-TT-binned_R3.01.txt). 83 δημοσιευμένα binned bandpowers, **ενσωματωμένα στο notebook** για offline χρήση.

In [29]:
PLANCK_DATA = r'''
47.711224 1479.33552 50.7654876 1461.11304
76.4716065 2034.96833 54.7101576 2062.38073
105.917385 2955.39416 64.9766440 2904.52526
135.605348 3869.51392 76.9143744 3902.48963
165.405597 4889.46506 86.5856259 4861.35537
195.266870 5464.10945 90.5533459 5534.80715
225.164945 5793.43954 87.1348811 5726.93135
255.086908 5372.88375 76.9383697 5374.98113
285.025248 4627.67753 62.5199141 4588.32266
314.975304 3604.23851 47.1589439 3596.89801
344.934027 2631.20029 33.8762191 2667.45188
374.899344 2033.05943 24.8190562 2018.16864
404.869791 1753.36253 20.7109083 1748.34025
434.844309 1787.57901 20.6713695 1821.92489
464.822111 2162.04649 22.7523861 2098.23728
494.802601 2422.08480 24.9726104 2397.72998
524.785320 2573.48050 25.9184118 2571.44487
554.769905 2546.29768 25.0510538 2551.80645
584.756069 2360.64530 22.7209283 2364.36140
614.743583 2095.43505 19.8505454 2104.80607
644.732258 1884.67698 17.5024864 1890.96016
674.721939 1813.16288 16.4306044 1812.52216
704.712498 1883.19393 16.7990643 1896.72650
734.703827 2097.13072 18.1645476 2102.25612
764.695836 2318.73584 19.7363538 2338.80907
794.688447 2464.58129 20.7062220 2504.17440
824.681596 2521.91260 20.5197620 2522.77302
854.675226 2394.12077 19.0381031 2371.41912
884.669287 2083.33867 16.5790348 2083.35227
914.663738 1740.72841 13.7374917 1732.53226
944.658541 1418.67342 11.1420483 1405.02882
974.653663 1172.95424 9.28646475 1170.14623
1004.64908 1062.40462 8.37987000 1058.87288
1034.64476 1047.86241 8.30074773 1059.36420
1064.64068 1132.35191 8.68575434 1126.69220
1094.63683 1211.01851 9.11274984 1202.98244
1124.63318 1231.87224 9.26050628 1238.58790
1154.62972 1205.78050 8.98528122 1208.04315
1184.62643 1117.92643 8.33171031 1115.02109
1214.62331 968.422243 7.54508665 986.438951
1244.62034 864.397867 6.64322644 859.469330
1274.61751 761.605936 5.98585735 766.845300
1304.61481 732.339633 5.68666016 725.319871
1334.61223 737.290327 5.71800485 731.898515
1364.60976 774.925851 5.94089388 767.047843
1394.60740 806.625346 6.17243469 803.588527
1424.60514 809.057064 6.25632882 816.784389
1454.60297 777.751089 6.11143430 792.658706
1484.60089 728.980580 5.74544667 731.226081
1514.59890 648.288650 5.24258624 644.929815
1544.59698 551.286111 4.72393254 552.838283
1574.59513 476.600995 4.30079600 473.410287
1604.59335 419.541270 4.04442416 418.888542
1634.59164 395.211496 3.96833591 392.152182
1664.58999 391.657847 4.03423921 387.313797
1694.58839 392.947519 4.17356282 393.097715
1724.58686 397.785903 4.31665510 397.391812
1754.58537 383.011743 4.41586492 391.427304
1784.58394 375.391110 4.45733427 371.983822
1814.58255 340.139516 4.45892926 341.550860
1844.58121 307.539268 4.45749277 306.532670
1874.57991 273.859536 4.49350451 274.153524
1904.57865 249.846901 4.59652807 250.006349
1934.57743 244.246300 4.77758093 236.236579
1964.57625 231.913142 5.02866666 231.349660
1994.57510 234.658998 5.42839879 231.209089
2024.57399 231.744942 5.86880339 230.808750
2054.57291 219.660112 6.20024969 226.094525
2084.57186 214.252650 6.53323083 215.102553
2114.57084 196.716713 6.87393629 198.340298
2144.56985 176.431323 7.23789713 178.237039
2174.56889 152.227062 7.64417922 158.049842
2204.56795 138.671109 8.10965997 140.733305
2234.56704 121.459219 8.64541703 128.004092
2264.56615 116.155153 9.25556670 120.051892
2294.56529 117.768892 9.93861339 115.705770
2324.56444 116.129438 10.6905048 113.100669
2354.56362 109.714703 11.5085752 110.345705
2384.56282 120.361413 12.3926722 106.139560
2414.56204 90.4515698 13.3433129 100.105444
2444.56128 103.080972 14.3819739 92.6666904
2474.56054 73.6743738 15.5113561 84.8106882
2499.02400 61.3290585 20.9398269 78.7736295
'''
planck = np.loadtxt(io.StringIO(PLANCK_DATA))
assert planck.shape == (83,4), planck.shape
ell, D_obs, D_err, D_best = planck.T
assert np.all(D_err>0) and np.all(np.diff(ell)>0)

def planck_lab(ell_max=1100):
    fig,ax=plt.subplots(figsize=(10.5,5))
    ax.plot(ell,D_best,color=COL['matter'],lw=2.3,label='Planck published best fit (binned)')
    ax.errorbar(ell,D_obs,yerr=D_err,fmt='.',ms=5,elinewidth=.8,alpha=.75,
                color=COL['dark'],ecolor=COL['other'],label='Planck 2018 TT measurements')
    ax.axvline(220,color=COL['radiation'],ls='--',alpha=.85,label=r'First peak $\ell\sim220$')
    ax.set(xlim=(40,ell_max),ylim=(0,6500),xlabel=r'Multipole $\ell$',
           ylabel=r'$D_\ell^{TT}$ [$\mu$K$^2$]',title='Observed CMB angular spectrum | Planck PR3')
    ax.legend(fontsize=9)
    fig.tight_layout();plt.show()
    print(f'Visible multipoles: 40–{ell_max}.  θ_min≈{180/ell_max:.3f}° | observational TT, not primordial tensor BB.')
lab(planck_lab,{'ell_max':widgets.IntSlider(value=1100,min=350,max=2500,step=50,description='max ℓ',continuous_update=False)} if HAS_WIDGETS else {}, {'ell_max':1100})

interactive(children=(IntSlider(value=1100, continuous_update=False, description='max ℓ', max=2500, min=350, s…

## 10 · Πόλωση του CMB: σχηματικά $E$ vs $B$
**E-mode:** radial/tangential προσανατολισμοί. **B-mode:** polarization lines με χαρακτηριστική περιστροφή $45^\circ$ στην απλή επίπεδη απεικόνιση.

**Scalar** $\to$ primordial $E$ (linear) · **Tensor** $\to E+B$ · **Lensing** επίσης $\to B$. **Το σχήμα δεν είναι πραγματικός χάρτης ή υπολογισμός $C_\ell^{BB}$.**

In [32]:
def eb_lab(pattern='E',scale=1.):
    coords=np.linspace(-1.4,1.4,13)
    X,Y=np.meshgrid(coords,coords)
    R=np.hypot(X,Y)
    alpha=np.arctan2(Y,X)+(np.pi/4 if pattern=='B' else 0)
    L=0.085*scale*(0.25+0.75*np.exp(-R**2/5))
    ux=L*np.cos(alpha);vy=L*np.sin(alpha)
    fig,ax=plt.subplots(figsize=(6.5,6))
    for x,y,u,v in zip(X.ravel(),Y.ravel(),ux.ravel(),vy.ravel()):
        ax.plot([x-u,x+u],[y-v,y+v],color=COL['matter'] if pattern=='E' else COL['tensor'],lw=1.7)
    ax.set(xlim=(-1.6,1.6),ylim=(-1.6,1.6),aspect='equal',title=f'Illustrative {pattern}-type polarization orientations')
    ax.set_axis_off();fig.tight_layout();plt.show()
lab(eb_lab,{'pattern':widgets.ToggleButtons(options=['E','B'],description='Pattern'),
            'scale':widgets.FloatSlider(value=1,min=.4,max=1.7,step=.1,description='Line size')} if HAS_WIDGETS else {}, {'pattern':'E','scale':1.})

interactive(children=(ToggleButtons(description='Pattern', options=('E', 'B'), value='E'), FloatSlider(value=1…

### Εξισώσεις που συνδέουν τα πάντα
$$
\boxed{\text{FLRW}\ \rightarrow\ \text{Inflation}\ \rightarrow\ \mathcal{P}_{\mathcal{R}}(k),\,\mathcal{P}_t(k)\ \rightarrow\ \text{CMB}\ \&\ \text{cosmic structure}}
$$
- $\mathcal{R}_k$ και $h_k$: σχεδόν σταθερά σε υπερ-Hubble κλίμακες υπό τις κατάλληλες συνθήκες.
- $\mathcal{P}_{\mathcal{R}}(k)$: scalar αρχικές συνθήκες · $\mathcal{P}_t(k)$: tensor αρχικές συνθήκες.
- $C_\ell^{TT}$: θερμοκρασιακές ανισοτροπίες · $C_\ell^{BB}$: πόλωση που μπορεί να περιλαμβάνει primordial tensor συνεισφορά.


